##### - Use this notebook to search for coincident (non-cloudy) Landsat8 and Sentinel-2 images
##### - The script exports respective Landsat8 and Sentinel-2 footprints for coincident images as shapefiles
##### - The footprints are indexed by observation date. 
##### - Use the following script '2-calculate-overlapping-footprints' to find dates and corresponding footprints with the highest portion of non-cloudy Sentinel-2/Landsat8 overlap. 

## 1.0 Libraries and directories

In [22]:
import ee 
import geemap
import geopandas as gpd
import datetime as dt
import pprint as pp
from shapely.geometry import shape

ee.Authenticate()
ee.Initialize(project='ee-green-by-another-name')

roi_name = 'AKCP'
cloud_threshold = 40

""" 
To select coincident images from a specific seasons, 
make list of timeframes with begining and end.
"""
season_start = '05-01' # 'MM-DD'
season_stop = '09-30'
# Using 2017, becuase SCL band for cloud shaddows, cirrus, etc isn't available until 2017-03-28
# https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED
year_start = 2017 # YYYY format
year_stop = 2025
if year_start != year_stop:
    year_range = range(year_start, year_stop)
else:
    year_range = [year_start]

timeframes = []
for year in year_range:
    begin = str(year) + '-' + season_start
    end = str(year) + '-' + season_stop
    tf = (begin, end)
    timeframes.append(tf)

sub_rois = gpd.read_file(f'./data/roi_shapes/rois/{roi_name}_sub_rois.shp')
sub_rois.head()

,id,sub_name,utm_zone,utm_epsg,geometry
0,1,AKCP_sub1,6N - row W,EPSG:32606,"POLYGON ((-149.97296 70.45217, -149.92228 70.4..."
1,2,AKCP_sub2,4N - row W,EPSG:32604,"POLYGON ((-157.4145 70.19548, -157.61719 70.11..."
2,3,AKCP_sub3,5N - row W,EPSG:32605,"POLYGON ((-151.83246 70.06075, -152.04983 70.0..."
3,4,AKCP_sub4,5N - row W,EPSG:32605,"POLYGON ((-154.46498 70.74353, -154.6626 70.67..."


## 3.0 Functions to search images and calculate footprints

In [23]:
def find_img_pairs(
        roi: ee.geometry, 
        start: str,
        end: str, 
        cloud_threshold: float):
    """
    Searches for images in the region and timeframe with a desired cloud threshold
    Returns a paired collection with images from an inner join by date
    """
    s2_string = 'COPERNICUS/S2_SR_HARMONIZED' # Use Level-2A data, because you need the SCL band for cirrus & cloud shaddow even on L-1C data
    ls8_string = 'LANDSAT/LC08/C02/T1_L2'
    # Produces image collection for all images within the roi and date range
    s2_col = (
        ee.ImageCollection(s2_string) 
        .filterDate(start, end)
        .filterBounds(roi)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_threshold))
    )

    ls8_col = (
        ee.ImageCollection(ls8_string)
        .filterDate(start, end)
        .filterBounds(roi)
        .filter(ee.Filter.lt('CLOUD_COVER', cloud_threshold))
    )

    def add_date_for_join(img: ee.image):
        """Formats the date consistently to join image collections"""
        return img.set('formatted_date', img.date().format('YYYYMMdd'))

    # Join image collections by date into a paired collection
    s2_col = s2_col.map(add_date_for_join)
    ls8_col = ls8_col.map(add_date_for_join)
    date_filter = ee.Filter.equals(leftField='formatted_date', rightField='formatted_date')
    inner_join = ee.Join.inner()
    paired_collection = inner_join.apply(s2_col, ls8_col, date_filter)

    return paired_collection

def separate_satellites(paired_fc: ee.featurecollection):
    """
    Makes two sepperate collections from the combined paired collections.
    This allows you to calculate each satellite's respective footprint on the coincindent date. 
    """
    s2_images = paired_fc.map(
        lambda feature: (
            ee.Image(feature.get('primary'))
                .set('formatted_date', ee.Image(feature.get('primary')).get('formatted_date'))
        )
    )
    s2_col = ee.ImageCollection(s2_images)

    ls8_images = paired_fc.map(
        lambda feature: (
            ee.Image(feature.get('secondary'))
                .set('formatted_date', ee.Image(feature.get('secondary')).get('formatted_date'))
        )
    )
    ls8_col = ee.ImageCollection(ls8_images)

    return s2_col, ls8_col

def mosiac_attrs_by_date(img_col, roi, satellite):
    """
    Uses the image collection of a given satellite (Sentinel-2 or Landsat8),
    Returns polygon feature collections with each date's corresponding non-cloudy footprint
    """

    # Using only one band to prevent wonky segmentation with the reduceToVectors() method
    if satellite == 'Sentinel-2':
        scale = 10
        select_band = 'B8'
    else:
        scale = 30
        select_band = 'SR_B5'
    
    distinct_dates = img_col.aggregate_array('formatted_date').distinct()

    def date_mosaic(date_str):
        """
        Makes an image mosiac for each date and returns a polygon with mosaic's footprint. 
        """
        date_str = ee.String(date_str)
        date_imgs = img_col.filter(ee.Filter.eq('formatted_date', date_str))
        mosaic = (date_imgs.mosaic()
                  .set('formatted_date', date_str)
                  .clip(roi)
                  .select(select_band))
        
        data_mask = mosaic.gt(0)
        # Fill random small holes in the data mask
        radius = 5 # pixels
        kernel = ee.Kernel.circle(radius, 'pixels')
        filled_data_mask = data_mask.focal_max(kernel=kernel).focal_min(kernel=kernel)


        polygon_boundaries = filled_data_mask.reduceToVectors(
            #geometry=roi,
            geometryType='polygon',
            scale=scale,
            maxPixels=1e13,
            eightConnected=True
        )

        # Take polygons with a large enough area, there's some random small polygons due to wonky segmentation
        polygons_with_area = polygon_boundaries.map(lambda f: f.set({
            'area_m2': f.geometry().area(maxError=1)
        }))
        large_enough = polygons_with_area.filter(ee.Filter.gt('area_m2', 100000))
        # Make polygons into a single geometry
        combined = large_enough.geometry(maxError=1)

        final_feature = ee.Feature(combined).set('formatted_date', date_str)
        
        return final_feature

    # Feature collection with each date's footprint
    footprints_fc = ee.FeatureCollection(distinct_dates.map(date_mosaic))
   
    return footprints_fc
    

## 4.0 Run the functions and produce footprints

In [24]:
def write_img_footprints(row: gpd.GeoSeries, timeframes: list):
    """
    Iterates through the list of time frames (e.g. May-Sep 2015 through May-Sep 2024)
    Runs the export function for each row (i.e., the sub roi)
    """

    geom = row.geometry
    roi_name = row['sub_name']
    coords = list(geom.exterior.coords)
    coords_list = [[x, y] for x, y in coords]
    roi = ee.Geometry.Polygon(coords_list)

    all_imgs_list = []

    for tf in timeframes:
        start = tf[0]
        end = tf[1]
        #print(f'---------  Observations for {tf}  ----------')
        paired = find_img_pairs(roi, start, end, cloud_threshold=cloud_threshold)
        all_imgs_list.append(paired)

    all_imgs = all_imgs_list[0]
    for i in range(1, len(all_imgs_list)):
        all_imgs = all_imgs.merge(all_imgs_list[i])

    s2, ls8 = separate_satellites(paired_fc=all_imgs)
    footprints_s2 = mosiac_attrs_by_date(s2, roi=roi, satellite='Sentinel-2')
    footprints_ls8 = mosiac_attrs_by_date(ls8, roi=roi, satellite='Landsat8')

    footprints_s2_filtered = footprints_s2.filterBounds(
        ee.Geometry.Rectangle([-180, -90, 180, 90])  # or your own region
    )
    # Start the export
    task_s2 = ee.batch.Export.table.toDrive(
        collection=footprints_s2_filtered,
        description=f'footprints_s2_roi_{roi_name}_years_{year_start}_{year_stop - 1}_dates_{season_start}_{season_stop}',
        folder='s2_roi_img_footprints',            
        fileNamePrefix=f'footprints_s2_roi_{roi_name}_years_{year_start}_{year_stop - 1}_dates_{season_start}_{season_stop}',      
        fileFormat='SHP',
    )
    task_s2.start()

    footprints_ls8_filtered = footprints_ls8.filterBounds(
        ee.Geometry.Rectangle([-180, -90, 180, 90])  # or your own region
    )
    # Start the export
    task_ls8 = ee.batch.Export.table.toDrive(
        collection=footprints_ls8_filtered,
        description=f'footprints_ls8_roi_{roi_name}_years_{year_start}_{year_stop - 1}_dates_{season_start}_{season_stop}',
        folder='ls8_roi_img_footprints',            
        fileNamePrefix=f'footprints_ls8_roi_{roi_name}_years_{year_start}_{year_stop - 1}_dates_{season_start}_{season_stop}',      
        fileFormat='SHP',
    )
    task_ls8.start()


In [25]:
for idx, row in sub_rois.iterrows():
    pp.pp(row)
    write_img_footprints(row, timeframes)
    print("Exporting")

id                                                          1
sub_name                                            AKCP_sub1
utm_zone                                           6N - row W
utm_epsg                                           EPSG:32606
geometry    POLYGON ((-149.9729605270181 70.45216740373714...
Name: 0, dtype: object
Exporting
id                                                          2
sub_name                                            AKCP_sub2
utm_zone                                           4N - row W
utm_epsg                                           EPSG:32604
geometry    POLYGON ((-157.41450460781704 70.1954845616769...
Name: 1, dtype: object
Exporting
id                                                          3
sub_name                                            AKCP_sub3
utm_zone                                           5N - row W
utm_epsg                                           EPSG:32605
geometry    POLYGON ((-151.832459729175 70.06074533150964,...
Name